# 📊 Benchmark Comparison: EfficientNet-B0 (CNN) vs. Hybrid CNN-ViT

Este notebook consolida los resultados de evaluación experimental sobre el **Conjunto de Test (Holdout 15% - 4.050 imágenes)**.

A diferencia de proyectos informales donde las métricas se copian a mano, este notebook **consume directamente los reportes serializados en JSON** generados por el pipeline de entrenamiento (`reports/test_report_*.json`).


In [ ]:
import sys
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
reports_dir = project_root / 'reports'


## 1. Carga Automatizada de Reportes Serializados (JSON)


In [ ]:
cnn_report = reports_dir / 'test_report_cnn.json'
hybrid_report = reports_dir / 'test_report_hybrid.json'

records = []

if cnn_report.exists():
    with open(cnn_report, 'r', encoding='utf-8') as f:
        data = json.load(f)
        records.append({
            'Modelo': 'EfficientNet-B0 (CNN)',
            'Tipo': 'Convolucional Puro',
            'Accuracy (%)': round(data['accuracy'] * 100, 2),
            'F1-Macro': round(data['f1_macro'], 4),
            'F1-Weighted': round(data['f1_weighted'], 4),
            'Test Loss': round(data['loss'], 4)
        })

if hybrid_report.exists():
    with open(hybrid_report, 'r', encoding='utf-8') as f:
        data = json.load(f)
        records.append({
            'Modelo': 'Hybrid CNN-ViT',
            'Tipo': 'Híbrido (CNN + Transformer)',
            'Accuracy (%)': round(data['accuracy'] * 100, 2),
            'F1-Macro': round(data['f1_macro'], 4),
            'F1-Weighted': round(data['f1_weighted'], 4),
            'Test Loss': round(data['loss'], 4)
        })

df = pd.DataFrame(records)
print("=" * 60)
print(" RESULTADOS EN EL CONJUNTO DE TEST (HOLDOUT 15%)")
print("=" * 60)
display(df)


## 2. Visualización Gráfica del Benchmark

Comparamos las métricas clave de ambas arquitecturas:


In [ ]:
if not df.empty:
    df_melt = df.melt(
        id_vars=['Modelo'],
        value_vars=['Accuracy (%)', 'F1-Macro', 'F1-Weighted'],
        var_name='Métrica',
        value_name='Puntuación'
    )

    plt.figure(figsize=(10, 6))
    sns.barplot(data=df_melt, x='Métrica', y='Puntuación', hue='Modelo', palette='Blues_d')
    plt.title('Comparación de Rendimiento en EuroSAT (Test Holdout)', fontsize=14, fontweight='bold')
    plt.ylabel('Puntuación')
    plt.ylim(0, 100)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.legend(title='Arquitectura')
    plt.tight_layout()
    plt.show()


## 3. Conclusiones Técnicas de Ingeniería

1. **Rigor Estadístico:** Ambas arquitecturas fueron evaluadas sobre exactamente el mismo conjunto intocable de 4.050 imágenes satelitales.
2. **Capacidad Convolucional vs. Atención Global:** EfficientNet-B0 proporciona una línea base sólida y eficiente gracias a su sesgo inductivo espacial, mientras que el módulo de autoatención del Vision Transformer agrega capacidad de modelado semántico global a través del token `[CLS]`.
3. **Reproducibilidad:** El flujo completo es reproducible desde la terminal mediante `python train.py` y desplegable con `python -m app.main`.
